In [ ]:
# Fase 3 baseline: duck harness (Tufa Labs) en la G4 — validación offline CORTA.
import json, os, pickle, subprocess, sys, sysconfig, time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1","true"}
NOTEBOOK_START = time.time()
os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"
# CUDA linker path para vLLM/torch en imagen Kaggle
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    e for e in ["/usr/local/nvidia/lib64", os.environ.get("LIBRARY_PATH","")] if e)
# Corte de validación offline (minutos) para NO gastar 9h de G4 cuando no es rerun.
OFFLINE_SOFT_MIN = float(os.environ.get("TAAF_OFFLINE_SOFT_MIN", "25"))
WORKING = Path("/kaggle/working"); WORKING.mkdir(parents=True, exist_ok=True)
print("TRUE_SUBMISSION =", TRUE_SUBMISSION)


In [ ]:
# Instalar arc-agi del wheelhouse de la competencia (offline)
COMP_ROOT = None
for dp, dn, _ in os.walk("/kaggle/input"):
    if "arc_agi_3_wheels" in dn:
        COMP_ROOT = Path(dp); break
assert COMP_ROOT, "wheelhouse no encontrado"
subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--no-index",
    "--no-warn-conflicts","--disable-pip-version-check",
    f"--find-links={COMP_ROOT/'arc_agi_3_wheels'}","arc-agi"], stdout=subprocess.DEVNULL)
import arc_agi; print("arc_agi OK")

# Localizar el bundle del solver por su marker
BUNDLE = None
for m in sorted(Path("/kaggle/input").rglob("taaf-kaggle-bundle.json")):
    try: _label = json.loads(m.read_text()).get("benchmark_label", "")
    except Exception: _label = ""
    if "anim" in _label:
        BUNDLE = m.parent; break
assert BUNDLE, "bundle TAAF no encontrado (adjunta jakobbrggen/taaf-kaggle-source-anim-20260807-anim)"
print("BUNDLE =", BUNDLE)

# Mapear datasets adjuntos a sus mounts
DATASET_SOURCES = ["jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
                   "driessmit1/arc3-vllm-h100-wheelhouse-v3",
                   "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot",
                   "thtennant/taaf-kaggle-source-share-fork"]
def mount(ref):
    o,s = ref.split("/",1)
    for c in (Path("/kaggle/input")/s, Path("/kaggle/input/datasets")/o/s):
        if c.exists(): return str(c)
    return str(Path("/kaggle/input")/s)
paths = {r: (str(BUNDLE) if i==0 else mount(r)) for i,r in enumerate(DATASET_SOURCES)}
paths["foysalemonshanto/qwen3-8-27b-fp8-repacked-v1"] = "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"
env_extra = {"TAAF_KAGGLE_INPUT_PATHS": json.dumps(paths, sort_keys=True),
             "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
             "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps([])}
os.environ.update(env_extra)
SETUP_ENV = WORKING/"taaf_setup_env.json"; SETUP_ENV.write_text(json.dumps(env_extra))
print(paths)


In [ ]:
# Importar repos del bundle y correr setup_commands (instala vLLM, arranca el server)
def source_entries(b):
    out=[]
    for repo in sorted((b/"src").iterdir(), reverse=True):
        for c in (repo/"src", repo):
            if c.is_dir(): out.append(c)
    return out
entries = source_entries(BUNDLE)
GRAFTS = None
for _g in Path("/kaggle/input").rglob("taaf-grafts/taaf_grafts/composite.py"):
    GRAFTS = _g.parent.parent; break
assert GRAFTS, "taaf-grafts no encontrado (adjunta thtennant/taaf-kaggle-source-share-fork)"
assert not (GRAFTS/"ARC3-Inference").exists(), "raiz de injertos demasiado ancha"
entries = entries + [GRAFTS]
print("GRAFTS =", GRAFTS)
for e in entries: sys.path.insert(0, str(e))
pth = Path(sysconfig.get_paths()["purelib"])/"taaf_sources.pth"
pth.write_text("".join(f"{e}\n" for e in entries))

def cmd_env():
    env = os.environ.copy(); env["PYTHON"]=sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"]=str(BUNDLE); env["TAAF_KAGGLE_WORKING_DIR"]=str(WORKING)
    env["TAAF_KAGGLE_SETUP_ENV"]=str(SETUP_ENV)
    env.update({str(k):str(v) for k,v in json.loads(SETUP_ENV.read_text()).items()})
    return env
env = cmd_env()
for c in json.loads((BUNDLE/"setup_commands.json").read_text()):
    # INYECCION MODELO QWEN3.8 (v19): mismas ruedas vLLM, otra identidad de modelo.
    _rep38 = [("MODEL_OWNER = 'driessmit1'", "MODEL_OWNER = 'foysalemonshanto'"),
              ("MODEL_SLUG = 'vrfai-qwen3-6-27b-fp8-hf-snapshot'",
               "MODEL_SLUG = 'qwen3-8-27b-fp8-repacked-v1'"),
              ("SERVED_MODEL_NAME = 'vrfai/Qwen3.6-27B-FP8'",
               "SERVED_MODEL_NAME = 'Qwen/Qwen3.8-27B-FP8'")]
    _n38 = 0
    for _viejo38, _nuevo38 in _rep38:
        if _viejo38 in c:
            c = c.replace(_viejo38, _nuevo38, 1); _n38 += 1
    if _n38:
        print(f"[model_qwen38] {_n38}/3 asignaciones reescritas", flush=True)
    print("setup:", c[:80], flush=True)
    subprocess.run(c, shell=True, check=True, cwd=WORKING, env=env)
    env = cmd_env(); os.environ.update(env)
for e in reversed([x for x in os.environ.get("PYTHONPATH","").split(os.pathsep) if x]):
    if e not in sys.path: sys.path.insert(0, e)
print("setup completo")


In [ ]:
# Cargar benchmark + target, jugar (offline recortado / gateway en rerun)
with open(BUNDLE/"deploy_target.pkl","rb") as f: target = pickle.load(f)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION
with open(BUNDLE/"benchmark_initial.pkl","rb") as f: bm = pickle.load(f)
bm.job_dir = WORKING; bm.n_passes = 1; bm.game_weights = None
os.environ.setdefault("RECORDINGS_DIR", str(WORKING/"server_recording"))
# GUARD DE BASE ANIM: sin esto el log no distingue "adjunte el bundle" de "las
# palancas del bundle estan encendidas". Los dos campos viajan dentro del pickle
# del solver (capturados del entorno del autor al desplegar), asi que se afirman
# aqui explicitamente en vez de confiar en el default.
for _flag in ("hard_noop_guard", "animation_awareness"):
    if not hasattr(bm.solver, _flag):
        raise RuntimeError("bundle equivocado: el solver no tiene " + _flag +
                           " (adjunta jakobbrggen/taaf-kaggle-source-anim-20260807-anim)")
    setattr(bm.solver, _flag, True)
import inference.utils.animation as _anim_mod
import inference.agent.noop_guard as _ng_mod
print("ANIM_BASE animation=%s noop_guard=%s hard_noop_guard=%s animation_awareness=%s" % (
    _anim_mod.__file__, _ng_mod.__file__,
    bm.solver.hard_noop_guard, bm.solver.animation_awareness))


# Graft install (base = bundle anim de jakobbrggen, el de LB-9) + schema_helpers:
# precarga helpers de analisis testeados (grid_diff, connected_components,
# action_effect_summary, recent_history) en el sandbox python del agente — el 27B
# reescribe esa plomeria con bugs en cada juego. NUESTRA tesis de feature injection,
# implementada por el autor del fork como graft sin habilitar (WP3).
# goalkeep NO va: marco 0.81 en el set oculto (-0.36 vs v12; ver working notes
# 2026-08-12). Blindado: cualquier fallo -> stock.
# Verificado en CPU local con scripts/smoke_graft_install.py (banner + prelude 8KB).
try:
    from taaf_grafts.composite import install as _graft_install
    _graft_install(bm, flags={"efficiency": True, "retry_guard": True,
                              "shortcircuit": False, "schema_helpers": True})
except Exception as exc:
    print(f"[taaf_grafts] graft failed, running stock: {type(exc).__name__}: {exc}")

# MAPA COGNITIVO en la costura C. El anfitrion mantiene el grafo de estados del
# nivel y lo resume en el prompt; el modelo no gasta ni un turno construyendolo.
#
# El registrador ENVUELVE el guardia de no-ops del harness en vez de observar los
# fotogramas por su cuenta: asi hereda la correccion de animacion (una accion que
# devolvio varios fotogramas NO es inerte aunque el tablero final sea identico) en
# lugar de repetir el error del viejo helper de navegacion, que comparaba solo el frame final.
try:
    import base64 as _b64m
    import taaf_grafts.schema_helpers as _shm
    import inference.agent.noop_guard as _ngm
    _nsm = {}
    exec(compile(_b64m.b64decode("IiIiTWFwYSBjb2duaXRpdm86IGdyYWZvIGRlIGVzdGFkb3MgY2FsY3VsYWRvIHBvciBlbCBBTkZJVFJJT04gZSBpbnllY3RhZG8gZW4gZWwgcHJvbXB0LgoKUE9SIFFVRQotLS0tLS0tCkVsIGluZm9ybWUgb2ZpY2lhbCBkZSBBUkMtQUdJLTMgbWlkZSBjdWF0cm8gY2FwYWNpZGFkZXM6IGV4cGxvcmFjaW9uLCBtb2RlbGFkbywKZmlqYWNpb24gZGUgbWV0YXMgeSBwbGFuaWZpY2FjaW9uLiBMb3MgZG9zIHByaW1lcm9zIHB1ZXN0b3MgZGVsICpwcmV2aWV3KiBubyBlcmFuCm1vZGVsb3MgZGUgbGVuZ3VhamU6IGVsIDJvIChCbGluZCBTcXVpcnJlbCwgNi43MSUpIGNvbnN0cnVpYSB1biAqKmdyYWZvIGRpcmlnaWRvIGRlCmVzdGFkb3MqKiBhIHBhcnRpciBkZSBsb3MgZm90b2dyYW1hcyB5IHBvZGFiYSBsYXMgYWNjaW9uZXMgcXVlIGhhY2lhbiBidWNsZSBvIG5vCmNhbWJpYWJhbiBuYWRhLiBMb3MgYWdlbnRlcyBkZSBsZW5ndWFqZSBjb24gdmlzaW9uIHNlIHF1ZWRhcm9uIGVuIDMuNzAtNC4zNyUgcG9ycXVlCiJjYXJlY2VuIGRlbCBzZWd1aW1pZW50byBkZSBlc3RhZG8gZm90b2dyYW1hIGEgZm90b2dyYW1hIHF1ZSBlc3RvcyBlbnRvcm5vcwpyZXF1aWVyZW4iLgoKRXMgdGFtYmllbiBsbyBxdWUgaGFjZSB1biBjZXJlYnJvOiBsYSB0ZW9yaWEgZGUgc2VnbWVudGFjaW9uIGRlIGV2ZW50b3MgZGljZSBxdWUgbm8Kc2UgY29kaWZpY2EgY2FkYSBmb3RvZ3JhbWEgc2lubyBsYSBUUkFOU0lDSU9OIGVuIGVsIGxpbWl0ZSBkZSBldmVudG8sIHkgbG9zIG1hcGFzCmNvZ25pdGl2b3MgYWxtYWNlbmFuICJxdWUgbGxldmEgYSBkb25kZSIuIE51ZXN0cm8gYWdlbnRlLCBlbiBjYW1iaW8sIHJlY2liZSB1bgp0cmFuc2NyaXB0byB5IHJlLWRlcml2YSBlbCBlc3RhZG8gY2FkYSB0dXJubyBjb24gMjcgbWlsIG1pbGxvbmVzIGRlIHBhcmFtZXRyb3MuCgpFTCBFUlJPUiBRVUUgRVNURSBNT0RVTE8gTk8gUkVQSVRFCi0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KYGBzcmMvYXJjMy9zYW5kYm94X25hdi5weTo6X25hdl9zaGlmdGBgIGNvbXBhcmFiYSBVTklDQU1FTlRFIGVsIGZvdG9ncmFtYSBmaW5hbC4gRW4KbG9zIGp1ZWdvcyBkZSB0aXBvIDEgKGZ0MDksIHNiMjYpIGVsIHByaW1lcm8geSBlbCB1bHRpbW8gc29uIGlkZW50aWNvcyBhdW5xdWUgbGEKYWNjaW9uIFNJIGhpem8gYWxnbywgYXNpIHF1ZSBhcXVlbCBoZWxwZXIgbGUgZW5zZW5hYmEgYWwgbW9kZWxvIHF1ZSBsYXMgYWNjaW9uZXMKaW5mb3JtYXRpdmFzIGVzdGFiYW4gbXVlcnRhcy4KCkFxdWkgbGFzIGFyaXN0YXMgTk8gc2UgZXRpcXVldGFuIGNvbXBhcmFuZG8gZm90b2dyYW1hcy4gU2UgdG9tYW4gZGVsIGd1YXJkaWEgZGUKbm8tb3BzIGRlbCBoYXJuZXNzIGFuaW0sIHF1ZSByZWNpYmUgcG9yIGFjY2lvbiBsYSB0dXBsYQpgYChuaXZlbCwgZmlybWFfYW50ZXMsIGZpcm1hX2FjY2lvbiwgYm9hcmRfY2hhbmdlZCwgYW5pbWF0ZWQpYGAg4oCUIGNvbiBgYGFuaW1hdGVkYGAKcHVlc3RvIGN1YW5kbyBlbCBlbnRvcm5vIGRldm9sdmlvIHZhcmlvcyBmb3RvZ3JhbWFzLiBVbmEgYWNjaW9uIGFuaW1hZGEgTlVOQ0Egc2UKcmVwb3J0YSBjb21vIGluZXJ0ZSwgYXVucXVlIGVsIHRhYmxlcm8gZmluYWwgc2VhIGlkZW50aWNvLiBMYSBjb3JyZWNjaW9uIHNlIGhlcmVkYQpkZWwgaGFybmVzcyBlbiB2ZXogZGUgcmUtaW1wbGVtZW50YXJzZS4KCkNPU1RFCi0tLS0tClNvbG8gdG9rZW5zIGRlIEVOVFJBREEgKGxhIG5vdGEgc2UgYW5hZGUgYWwgcHJvbXB0IGRlbCB1c3VhcmlvKS4gQ0VSTyBlc2NyaXR1cmEKZXhpZ2lkYSBhbCBtb2RlbG86IGxvcyBkb3MgcGFyY2hlcyBhbnRlcmlvcmVzIHF1ZSBleGlnaWFuIHF1ZSBlbCBtb2RlbG8gZXNjcmliaWVyYQoocmFudXJhcywgbW9kZWxvIGRlIG11bmRvKSBjb3N0YXJvbiBhY2Npb25lcyB5IG5vIHBhZ2Fyb24uCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKTUFYX0xJU1RBID0gNiAgICAgICAgICAjIGN1YW50b3MgZWxlbWVudG9zIGNvbW8gbXVjaG8gcG9yIGxpbmVhIGRlIGxhIG5vdGEKTUFYX0NFTERBU19DTElDSyA9IDggICAjIGNlbGRhcyBkZSBjbGljayBxdWUgc2UgZW51bWVyYW4gYW50ZXMgZGUgcmVzdW1pciBhIGNhamEKCgpjbGFzcyBNYXBSZWNvcmRlcjoKICAgICIiIkVudm9sdG9yaW8gZGVsIGBgTm9vcEd1YXJkYGAgZGVsIGhhcm5lc3MgcXVlIGFkZW1hcyByZWdpc3RyYSBlbCBncmFmby4KCiAgICBTZSBpbnN0YWxhIGVuIGx1Z2FyIGRlbCBndWFyZGlhIHJlYWw6IGRlbGVnYSBUT0RPIGVuIGVsIChwYXJhIG5vIGFsdGVyYXIgc3UKICAgIGNvbXBvcnRhbWllbnRvIG5pIHVuYSBjb21hKSB5IGRlIHBhc28gYXB1bnRhIGNhZGEgdHJhbnNpY2lvbiBvYnNlcnZhZGEuCgogICAgTGEgYXJpc3RhIGkgdmEgZGVsIGVzdGFkbyBgYGJlZm9yZV9zaWdgYCBkZSBsYSBvYnNlcnZhY2lvbiBpIGFsIGVzdGFkbwogICAgYGBiZWZvcmVfc2lnYGAgZGUgbGEgb2JzZXJ2YWNpb24gaSsxIGRlbCBtaXNtbyBuaXZlbDogZWwgaGFybmVzcyByZWFzaWduYQogICAgYGBub29wX2d1YXJkX2JvYXJkX3NpZ2BgIGNvbiBlbCB0YWJsZXJvIHJlZnJlc2NhZG8ganVzdG8gZGVzcHVlcyBkZSBvYnNlcnZhciwKICAgIGFzaSBxdWUgbGFzIG9ic2VydmFjaW9uZXMgY29uc2VjdXRpdmFzIGVuY2FkZW5hbiBlbCByZWNvcnJpZG8gcmVhbC4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbm5lcik6CiAgICAgICAgc2VsZi5faW5uZXIgPSBpbm5lcgogICAgICAgIHNlbGYucmVnaXN0cm9zOiBsaXN0W2RpY3RdID0gW10KCiAgICAjIC0tIGRlbGVnYWNpb24gdG90YWwgYWwgZ3VhcmRpYSByZWFsIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCiAgICBkZWYgX19nZXRhdHRyX18oc2VsZiwgbmFtZSk6CiAgICAgICAgcmV0dXJuIGdldGF0dHIoc2VsZi5faW5uZXIsIG5hbWUpCgogICAgZGVmIGlzX2tub3duX25vb3Aoc2VsZiwgbGV2ZWwsIGJvYXJkX3NpZywgYWN0aW9uX3NpZyk6CiAgICAgICAgcmV0dXJuIHNlbGYuX2lubmVyLmlzX2tub3duX25vb3AobGV2ZWwsIGJvYXJkX3NpZywgYWN0aW9uX3NpZykKCiAgICBkZWYgb2JzZXJ2ZShzZWxmLCAqLCBsZXZlbCwgYm9hcmRfYmVmb3JlX3NpZywgYWN0aW9uX3NpZywgYm9hcmRfY2hhbmdlZCwKICAgICAgICAgICAgICAgIGFuaW1hdGVkPUZhbHNlLCAqKmt3KToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHNlbGYucmVnaXN0cm9zLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAibGV2ZWwiOiBfZW50ZXJvKGxldmVsKSwKICAgICAgICAgICAgICAgICJhbnRlcyI6IHN0cihib2FyZF9iZWZvcmVfc2lnKSwKICAgICAgICAgICAgICAgICJhY2Npb24iOiAiICIuam9pbihzdHIoYWN0aW9uX3NpZyBvciAiIikuc3BsaXQoKSksCiAgICAgICAgICAgICAgICAjICJoaXpvIGFsZ28iID0gY2FtYmlvIGVsIHRhYmxlcm8gTyBkZXZvbHZpbyBhbmltYWNpb24uIExhIHNlZ3VuZGEKICAgICAgICAgICAgICAgICMgbWl0YWQgZXMgbGEgcXVlIGV2aXRhIGVsIGVycm9yIGRlIG5hdi4KICAgICAgICAgICAgICAgICJlZmVjdG8iOiBib29sKGJvYXJkX2NoYW5nZWQpIG9yIGJvb2woYW5pbWF0ZWQpLAogICAgICAgICAgICAgICAgInNvbG9fYW5pbWFjaW9uIjogYm9vbChhbmltYXRlZCkgYW5kIG5vdCBib29sKGJvYXJkX2NoYW5nZWQpLAogICAgICAgICAgICB9KQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246ICAjIG5vcWE6IEJMRTAwMSDigJQgcmVnaXN0cmFyIGphbWFzIHB1ZWRlIHR1bWJhciBlbCBndWFyZGlhCiAgICAgICAgICAgIHBhc3MKICAgICAgICByZXR1cm4gc2VsZi5faW5uZXIub2JzZXJ2ZSgKICAgICAgICAgICAgbGV2ZWw9bGV2ZWwsIGJvYXJkX2JlZm9yZV9zaWc9Ym9hcmRfYmVmb3JlX3NpZywgYWN0aW9uX3NpZz1hY3Rpb25fc2lnLAogICAgICAgICAgICBib2FyZF9jaGFuZ2VkPWJvYXJkX2NoYW5nZWQsIGFuaW1hdGVkPWFuaW1hdGVkLCAqKmt3KQoKICAgIGRlZiByZXNldChzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYucmVnaXN0cm9zID0gW10KCgpkZWYgX2VudGVybyh2LCBkZWZlY3RvPTApOgogICAgdHJ5OgogICAgICAgIHJldHVybiBpbnQodikKICAgIGV4Y2VwdCAoVHlwZUVycm9yLCBWYWx1ZUVycm9yKToKICAgICAgICByZXR1cm4gZGVmZWN0bwoKCmRlZiBfbm9tYnJlKGFjY2lvbjogc3RyKSAtPiBzdHI6CiAgICAiIiInTU9VU0Uocm93PTIwLCBjb2w9MTMpJyAtPiAnTU9VU0UnOyAgJ1VQJyAtPiAnVVAnLiIiIgogICAgcmV0dXJuIChhY2Npb24gb3IgIiIpLnNwbGl0KCIoIilbMF0uc3RyaXAoKS51cHBlcigpCgoKZGVmIF9jZWxkYShhY2Npb246IHN0cik6CiAgICAiIiJFeHRyYWUgKGZpbGEsIGNvbCkgZGUgdW5hIGFjY2lvbiBkZSByYXRvbiwgbyBOb25lLiIiIgogICAgdHh0ID0gYWNjaW9uIG9yICIiCiAgICBpZiAicm93PSIgbm90IGluIHR4dCBvciAiY29sPSIgbm90IGluIHR4dDoKICAgICAgICByZXR1cm4gTm9uZQogICAgdHJ5OgogICAgICAgIGZpbGEgPSBpbnQodHh0LnNwbGl0KCJyb3c9IiwgMSlbMV0uc3BsaXQoIiwiLCAxKVswXS5zdHJpcCgiKSAiKSkKICAgICAgICBjb2wgPSBpbnQodHh0LnNwbGl0KCJjb2w9IiwgMSlbMV0uc3BsaXQoIiwiLCAxKVswXS5zdHJpcCgiKSAiKSkKICAgICAgICByZXR1cm4gZmlsYSwgY29sCiAgICBleGNlcHQgKFZhbHVlRXJyb3IsIEluZGV4RXJyb3IpOgogICAgICAgIHJldHVybiBOb25lCgoKZGVmIGNvbnN0cnVpcl9ncmFmbyhyZWdpc3Ryb3M6IGxpc3RbZGljdF0sIG5pdmVsOiBpbnQpIC0+IGRpY3Q6CiAgICAiIiJHcmFmbyBkZWwgbml2ZWwgaW5kaWNhZG86IGFyaXN0YXMsIGVzdGFkb3MgeSBhZm9yZGFuY2lhcy4iIiIKICAgIGRlbG5pdmVsID0gW3IgZm9yIHIgaW4gcmVnaXN0cm9zIGlmIHIuZ2V0KCJsZXZlbCIpID09IG5pdmVsXQogICAgYXJpc3RhczogbGlzdFt0dXBsZVtzdHIsIHN0ciwgc3RyLCBib29sXV0gPSBbXSAgICMgKG9yaWdlbiwgYWNjaW9uLCBkZXN0aW5vLCBlZmVjdG8pCiAgICBmb3IgaSwgciBpbiBlbnVtZXJhdGUoZGVsbml2ZWwpOgogICAgICAgIGRlc3Rpbm8gPSBkZWxuaXZlbFtpICsgMV1bImFudGVzIl0gaWYgaSArIDEgPCBsZW4oZGVsbml2ZWwpIGVsc2UgTm9uZQogICAgICAgIGFyaXN0YXMuYXBwZW5kKChyWyJhbnRlcyJdLCByWyJhY2Npb24iXSwgZGVzdGlubywgclsiZWZlY3RvIl0pKQoKICAgIGVzdGFkb3M6IGRpY3Rbc3RyLCBpbnRdID0ge30KICAgIGZvciByIGluIGRlbG5pdmVsOgogICAgICAgIGVzdGFkb3NbclsiYW50ZXMiXV0gPSBlc3RhZG9zLmdldChyWyJhbnRlcyJdLCAwKSArIDEKCiAgICB1dGlsZXM6IHNldFtzdHJdID0gc2V0KCkKICAgIGluZXJ0ZXM6IGRpY3Rbc3RyLCBpbnRdID0ge30KICAgIGNlbGRhc191dGlsZXM6IGxpc3RbdHVwbGVbaW50LCBpbnRdXSA9IFtdCiAgICBzb2xvX2FuaW0gPSAwCiAgICBmb3IgciBpbiBkZWxuaXZlbDoKICAgICAgICBuID0gX25vbWJyZShyWyJhY2Npb24iXSkKICAgICAgICBpZiByWyJlZmVjdG8iXToKICAgICAgICAgICAgdXRpbGVzLmFkZChuKQogICAgICAgICAgICBjID0gX2NlbGRhKHJbImFjY2lvbiJdKQogICAgICAgICAgICBpZiBjIGFuZCBjIG5vdCBpbiBjZWxkYXNfdXRpbGVzOgogICAgICAgICAgICAgICAgY2VsZGFzX3V0aWxlcy5hcHBlbmQoYykKICAgICAgICBlbHNlOgogICAgICAgICAgICBpbmVydGVzW25dID0gaW5lcnRlcy5nZXQobiwgMCkgKyAxCiAgICAgICAgaWYgci5nZXQoInNvbG9fYW5pbWFjaW9uIik6CiAgICAgICAgICAgIHNvbG9fYW5pbSArPSAxCgogICAgcmV0dXJuIHsKICAgICAgICAiYXJpc3RhcyI6IGFyaXN0YXMsCiAgICAgICAgImVzdGFkb3MiOiBlc3RhZG9zLAogICAgICAgICJ1dGlsZXMiOiB1dGlsZXMsCiAgICAgICAgIyBpbmVydGUgZGUgdmVyZGFkID0gbnVuY2EgdHV2byBlZmVjdG8gZW4gTklOR1VOIGVzdGFkbyBkZSBlc3RlIG5pdmVsCiAgICAgICAgIm51bmNhX3V0aWxlcyI6IHtuIGZvciBuIGluIGluZXJ0ZXMgaWYgbiBub3QgaW4gdXRpbGVzfSwKICAgICAgICAiY2VsZGFzX3V0aWxlcyI6IGNlbGRhc191dGlsZXMsCiAgICAgICAgInNvbG9fYW5pbWFjaW9uIjogc29sb19hbmltLAogICAgICAgICJuX2FjY2lvbmVzIjogbGVuKGRlbG5pdmVsKSwKICAgIH0KCgpkZWYgcmVuZGVyX21hcF9ub3RlKHJlZ2lzdHJvczogbGlzdFtkaWN0XSwgbml2ZWw6IGludCwgZmlybWFfYWN0dWFsOiBzdHIgfCBOb25lLAogICAgICAgICAgICAgICAgICAgIGFjY2lvbmVzX3ZhbGlkYXM6IGxpc3Rbc3RyXSB8IE5vbmUpIC0+IHN0cjoKICAgICIiIk5vdGEgY29tcGFjdGEgcGFyYSBlbCBwcm9tcHQuIERldnVlbHZlICcnIHNpIG5vIGhheSBuYWRhIHF1ZSBhcG9ydGFyLiIiIgogICAgaWYgbm90IHJlZ2lzdHJvczoKICAgICAgICByZXR1cm4gIiIKICAgIGcgPSBjb25zdHJ1aXJfZ3JhZm8ocmVnaXN0cm9zLCBuaXZlbCkKICAgIGlmIG5vdCBnWyJhcmlzdGFzIl06CiAgICAgICAgcmV0dXJuICIiCgogICAgbGluZWFzOiBsaXN0W3N0cl0gPSBbXQogICAgbl9lc3RhZG9zID0gbGVuKGdbImVzdGFkb3MiXSkKICAgIHZpc2l0YXMgPSBnWyJlc3RhZG9zIl0uZ2V0KGZpcm1hX2FjdHVhbCBvciAiIiwgMCkKCiAgICBjYWIgPSAoZiJNQVBBIERFTCBBTkZJVFJJT04gKG5pdmVsIHtuaXZlbH0sIHtnWyduX2FjY2lvbmVzJ119IGFjY2lvbmVzIGFxdWksICIKICAgICAgICAgICBmIntuX2VzdGFkb3N9IGVzdGFkb3MgZGlzdGludG9zKSIpCiAgICBsaW5lYXMuYXBwZW5kKGNhYikKCiAgICAjIDEuIFF1ZSBzZSBoYSBwcm9iYWRvIERFU0RFIGVsIGVzdGFkbyBhY3R1YWwsIHkgY29uIHF1ZSByZXN1bHRhZG8uCiAgICBpZiBmaXJtYV9hY3R1YWw6CiAgICAgICAgZGVzZGUgPSBbKGEsIGRzdCwgZWYpIGZvciAob3JnLCBhLCBkc3QsIGVmKSBpbiBnWyJhcmlzdGFzIl0gaWYgb3JnID09IGZpcm1hX2FjdHVhbF0KICAgICAgICBpZiBkZXNkZToKICAgICAgICAgICAgdmlzdG9zOiBkaWN0W3N0ciwgYm9vbF0gPSB7fQogICAgICAgICAgICBmb3IgYSwgX2RzdCwgZWYgaW4gZGVzZGU6CiAgICAgICAgICAgICAgICB2aXN0b3NbYV0gPSB2aXN0b3MuZ2V0KGEsIEZhbHNlKSBvciBlZgogICAgICAgICAgICBoZWNoYXMgPSBbZiJ7YX17JycgaWYgZWYgZWxzZSAnIChzaW4gZWZlY3RvKSd9IgogICAgICAgICAgICAgICAgICAgICAgZm9yIGEsIGVmIGluIGxpc3QodmlzdG9zLml0ZW1zKCkpWzpNQVhfTElTVEFdXQogICAgICAgICAgICBsaW5lYXMuYXBwZW5kKGYiLSBkZXNkZSBlc3RlIGVzdGFkbyB5YSBwcm9iYXN0ZTogeycsICcuam9pbihoZWNoYXMpfSIpCiAgICAgICAgaWYgdmlzaXRhcyA+IDE6CiAgICAgICAgICAgIGxpbmVhcy5hcHBlbmQoZiItIEFURU5DSU9OOiBlc3RhcyBlbiB1biBlc3RhZG8geWEgdmlzaXRhZG8ge3Zpc2l0YXN9IHZlY2VzICIKICAgICAgICAgICAgICAgICAgICAgICAgICBmIihwb3NpYmxlIGJ1Y2xlKSIpCgogICAgICAgICMgMi4gTGEgZnJvbnRlcmE6IGFjY2lvbmVzIHZhbGlkYXMgbm8gcHJvYmFkYXMgQVFVSSB5IHF1ZSBhZGVtYXMgbm8gZXN0YW4KICAgICAgICAjICAgIHlhIGRlbW9zdHJhZGFzIGluZXJ0ZXMgZW4gdG9kbyBlbCBuaXZlbC4gU2luIGVzdGUgc2VndW5kbyBmaWx0cm8gbGEKICAgICAgICAjICAgIG5vdGEgc2UgY29udHJhZGljZSBzb2xhIChsaXN0YXJpYSB1bmEgYWNjaW9uIGNvbW8gInBvciBwcm9iYXIiIHkgZG9zCiAgICAgICAgIyAgICBsaW5lYXMgbWFzIGFiYWpvIGNvbW8gIm51bmNhIGhpem8gbmFkYSIpLgogICAgICAgIGlmIGFjY2lvbmVzX3ZhbGlkYXM6CiAgICAgICAgICAgIHByb2JhZGFzID0ge19ub21icmUoYSkgZm9yIChvcmcsIGEsIF9kLCBfZSkgaW4gZ1siYXJpc3RhcyJdIGlmIG9yZyA9PSBmaXJtYV9hY3R1YWx9CiAgICAgICAgICAgIGZyb250ZXJhID0gW2EgZm9yIGEgaW4gYWNjaW9uZXNfdmFsaWRhcwogICAgICAgICAgICAgICAgICAgICAgICBpZiBhLnVwcGVyKCkgbm90IGluIHByb2JhZGFzCiAgICAgICAgICAgICAgICAgICAgICAgIGFuZCBhLnVwcGVyKCkgbm90IGluIGdbIm51bmNhX3V0aWxlcyJdCiAgICAgICAgICAgICAgICAgICAgICAgIGFuZCBhLnVwcGVyKCkgIT0gIk1PVVNFIl0KICAgICAgICAgICAgaWYgZnJvbnRlcmE6CiAgICAgICAgICAgICAgICBsaW5lYXMuYXBwZW5kKGYiLSBkZXNkZSBlc3RlIGVzdGFkbyBOTyBoYXMgcHJvYmFkbzogIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmInsnLCAnLmpvaW4oZnJvbnRlcmFbOk1BWF9MSVNUQV0pfSIpCgogICAgIyAzLiBBZm9yZGFuY2lhcyBkZWwgbml2ZWw6IHF1ZSBoYSBzZXJ2aWRvIHkgcXVlIG5vLCBudW5jYS4KICAgIGlmIGdbIm51bmNhX3V0aWxlcyJdOgogICAgICAgIGxpbmVhcy5hcHBlbmQoZiItIG51bmNhIGhhbiBoZWNobyBuYWRhIGVuIGVzdGUgbml2ZWw6ICIKICAgICAgICAgICAgICAgICAgICAgIGYieycsICcuam9pbihzb3J0ZWQoZ1snbnVuY2FfdXRpbGVzJ10pWzpNQVhfTElTVEFdKX0iKQogICAgaWYgZ1siY2VsZGFzX3V0aWxlcyJdOgogICAgICAgIGNzID0gZ1siY2VsZGFzX3V0aWxlcyJdCiAgICAgICAgaWYgbGVuKGNzKSA8PSBNQVhfQ0VMREFTX0NMSUNLOgogICAgICAgICAgICB0ZXh0byA9ICIsICIuam9pbihmIih7Zn0se2N9KSIgZm9yIGYsIGMgaW4gY3MpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZnMgPSBbZiBmb3IgZiwgXyBpbiBjc10KICAgICAgICAgICAgY29scyA9IFtjIGZvciBfLCBjIGluIGNzXQogICAgICAgICAgICB0ZXh0byA9IChmIntsZW4oY3MpfSBjZWxkYXMgZW4gZmlsYXMge21pbihmcyl9LXttYXgoZnMpfSwgIgogICAgICAgICAgICAgICAgICAgICBmImNvbHMge21pbihjb2xzKX0te21heChjb2xzKX0iKQogICAgICAgIGxpbmVhcy5hcHBlbmQoZiItIGNsaWNrcyBxdWUgU0kgaGljaWVyb24gYWxnbzoge3RleHRvfSIpCgogICAgIyA0LiBMYSBjb3JyZWNjaW9uIGRlIGFuaW1hY2lvbiwgZGljaGEgZXhwbGljaXRhbWVudGUuCiAgICBpZiBnWyJzb2xvX2FuaW1hY2lvbiJdOgogICAgICAgIGxpbmVhcy5hcHBlbmQoZiItIHtnWydzb2xvX2FuaW1hY2lvbiddfSBhY2Npb24oZXMpIHR1dmllcm9uIGVmZWN0byBTT0xPIGVuIGxhICIKICAgICAgICAgICAgICAgICAgICAgIGYiYW5pbWFjaW9uICh0YWJsZXJvIGZpbmFsIGlkZW50aWNvKTogTk8gc29uIGluZXJ0ZXMiKQoKICAgIGlmIGxlbihsaW5lYXMpID09IDE6CiAgICAgICAgcmV0dXJuICIiCiAgICByZXR1cm4gIlxuIi5qb2luKGxpbmVhcykK").decode("utf-8"), "cognitive_map.py", "exec"), _nsm)
    _MapRecorder = _nsm["MapRecorder"]
    _render_map_note = _nsm["render_map_note"]

    _orig_ensure_m = _shm.SchemaHelpersToolAgent._ensure_session
    _orig_bup_m = _shm.SchemaHelpersToolAgent._build_user_prompt

    def _ensure_with_map(self, state_path):
        _orig_ensure_m(self, state_path)
        try:
            g = getattr(self, "_noop_guard", None)
            if g is not None and not isinstance(g, _MapRecorder):
                self._noop_guard = _MapRecorder(g)
        except Exception:
            pass

    def _bup_with_map(self, action_num, **kw):
        base = _orig_bup_m(self, action_num, **kw)
        try:
            rec = getattr(self, "_noop_guard", None)
            if not isinstance(rec, _MapRecorder):
                return base
            fr = kw.get("current_frame")
            nivel = fr.level if fr is not None else 1
            firma = _ngm.board_signature(fr.grid) if fr is not None else None
            nota = _render_map_note(rec.registros, nivel, firma, kw.get("valid_actions"))
        except Exception:
            return base
        return base + "\n" + nota if nota else base

    _shm.SchemaHelpersToolAgent._ensure_session = _ensure_with_map
    _shm.SchemaHelpersToolAgent._build_user_prompt = _bup_with_map
    print("COGNITIVE_MAP injected on seam C:", len(_nsm), "symbols")
except Exception as exc:
    print("[cognitive_map] injection failed, running stock: %s: %s" % (type(exc).__name__, exc))

import arc_agi, taaf.game_api
def games_offline(d):
    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=d)
    ar = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=d)
    return [taaf.game_api.GameAPI(env_name=e.game_id, arcade_spec=spec) for e in ar.available_environments]
def games_comp():
    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.COMPETITION,
                                    arc_base_url=os.environ["ARC_BASE_URL"], environments_dir="")
    ar = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.COMPETITION,
                        arc_base_url=spec.arc_base_url, environments_dir="")
    return [taaf.game_api.GameAPI(env_name=e.game_id, arcade_spec=spec) for e in ar.available_environments]

soft_end = None
if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY","test-key-123")
    os.environ.setdefault("ARC_BASE_URL","http://gateway:8001/")
    dl = time.monotonic()+600
    while time.monotonic()<dl:
        try:
            with urlopen(os.environ["ARC_BASE_URL"]+"api/games", timeout=10) as r:
                if r.status<500: break
        except Exception: pass
        time.sleep(5)
    bm.games = games_comp()
else:
    bm.games = games_offline(str(COMP_ROOT/"environment_files"))
    soft_end = datetime.fromtimestamp(NOTEBOOK_START)+timedelta(minutes=OFFLINE_SOFT_MIN)

import pandas as pd
pd.DataFrame([["1_0","1",True,1]], columns=["row_id","game_id","end_of_game","score"]).to_parquet(WORKING/"submission.parquet", index=False)

try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
finally:
    for c in json.loads((BUNDLE/"teardown_commands.json").read_text()):
        subprocess.run(c, shell=True, check=False, cwd=WORKING, env=cmd_env())
print("run terminado")
